# Train Seq2Seq Cross-Attention Classifier - StereoQueerEval Task A

Binary stereotype detection (`yes` / `no`) with `src/train_seq2seq.py`.

### Architecture
- **Context (Video Title + Description)** feeds into the **Pretrained Encoder** as memory.
- **YouTube Comment** feeds into the **Pretrained Decoder** as `decoder_input_ids`.
- Pretrained **cross-attention** in every decoder layer queries the context representation without training scratch transformer blocks from random weights.
- Output representation is extracted at the comment's `</s>` (EOS) token (or mean-pooled) and classified via a 2-way classification head (`no` / `yes`).

### Key Design Choices
- **Backbones**: Pretrained Multilingual Seq2Seq models:
  - `facebook/mbart-large-50` (mBART-50, ~610M params)
  - `facebook/nllb-200-distilled-600M` (Meta NLLB-200, ~620M params)
  - `facebook/bart-base` (~139M params, English only, fast smoke tests)
- **Language-aware tokenization**: Automatically prepends appropriate language tags (`en_XX`, `it_IT`, `nl_XX` for mBART; `eng_Latn`, `ita_Latn`, `nld_Latn` for NLLB).
- **Causal Decoder Handling**: In standard decoders, token $i$ only attends to tokens $0 \dots i$. We extract the final `</s>` (EOS) token representation, which has seen the entire comment and cross-attended to all video context.
- **Memory efficiency on Kaggle GPUs**: Uses `--batch-size 4 --grad-accum 4` (effective batch 16) with FP16 mixed precision to comfortably fit within 16GB VRAM (T4 / P100).


In [ ]:
import os, subprocess, torch

if not os.path.isdir('Try'):
    subprocess.run('git clone https://github.com/AmnO-O/Try.git', shell=True, check=True)
os.chdir('Try')
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print('Cwd:', os.getcwd())
print('src/train_seq2seq.py exists:', os.path.exists('src/train_seq2seq.py'))
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))


## 1. Locate the data

Finds `StereoQueerEval_*_training.tsv` automatically across Kaggle input directories, Colab, or local folders, and exports `LGBT_DATA_DIR` and `LGBT_WEIGHTS_DIR`.


In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.environ['KAGGLE_INPUT_PATH'])
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return hits
    return []

hits = _first(candidates)
if not hits:
    raise RuntimeError('StereoQueerEval_*_training.tsv not found - mount the dataset first')
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs(os.environ['LGBT_WEIGHTS_DIR'], exist_ok=True)
print('Data dir:', os.environ['LGBT_DATA_DIR'])
print('Weights: ', os.environ['LGBT_WEIGHTS_DIR'])
print('Files:   ', [os.path.basename(p) for p in hits])
assert len(hits) >= 1


## 2. Sanity check the dataset split

Verifies row counts, label distribution, and video-level grouping for **EN / IT / NL**.
Video-level split ensures comments belonging to the same YouTube video do not leak across train and validation.


In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod

LANGS = ['EN', 'IT', 'NL']
df = data_mod.load_frame(languages=LANGS)
print('Total rows:', len(df), '| By language:', df['lang'].value_counts().to_dict())
tr, va = data_mod.split_by_video(df, val_fraction=0.1, seed=42)
print('Train / Val rows:', len(tr), '/', len(va))
print('Video title overlap:', len(set(tr['yt_title']) & set(va['yt_title'])))


## 3. Train Seq2Seq Cross-Attention Classifier (`facebook/mbart-large-50`)

We train `facebook/mbart-large-50`:
- Video context (title + description) feeds into the encoder (kept frozen with `--unfreeze-enc 0` to preserve robust memory representations).
- Comments feed into the decoder, with top 2 decoder layers unfrozen (`--unfreeze-dec 2`) to specialize cross-attention to YouTube comments.
- `--grad-accum 4` with `--batch-size 4` gives an effective batch size of 16 while fitting within 16GB GPU memory.
- Differential learning rates: `1e-5` for backbone decoder blocks, `3e-4` for the classification head.


In [ ]:
!python src/train_seq2seq.py \
    --backbone facebook/mbart-large-50 \
    --languages EN IT NL \
    --epochs 15 \
    --patience 4 \
    --batch-size 4 \
    --grad-accum 4 \
    --unfreeze-enc 0 \
    --unfreeze-dec 2 \
    --enc-lr 1e-5 \
    --dec-lr 1e-5 \
    --head-lr 3e-4 \
    --pooling eos \
    --context-max-len 256 \
    --comment-max-len 128 \
    --out weights/best_mbart_stereotype.pt


## 4. Plot Training & Validation Curves

Visualizes train loss, validation loss, and validation Macro-F1 across epochs.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history_file = 'weights/best_mbart_stereotype_history.csv'
if os.path.exists(history_file):
    h = pd.read_csv(history_file)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(h['epoch'], h['train_loss'], '--', label='train')
    ax[0].plot(h['epoch'], h['val_loss'], '-', label='val')
    ax[0].set_title('Cross-Entropy Loss'); ax[0].grid(alpha=0.3); ax[0].legend()
    
    ax[1].plot(h['epoch'], h['val_macro_f1'], '-o', color='green', ms=4, label='val macro-F1')
    ax[1].set_title('Validation Macro-F1'); ax[1].grid(alpha=0.3); ax[1].legend()
    plt.tight_layout(); plt.show()
    
    best = h.loc[h['val_macro_f1'].idxmax()]
    print(f"Best Epoch: {int(best['epoch'])} | Macro-F1: {best['val_macro_f1']:.4f} | Accuracy: {best['val_acc']:.4f}")
else:
    print(f"{history_file} not found. Run Section 3 first.")


## 5. Evaluate Best Checkpoint & Per-Language Breakdown

Loads `weights/best_mbart_stereotype.pt` and evaluates across the validation set and each language (EN, IT, NL).


In [ ]:
import torch
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import model_seq2seq
import train_seq2seq as tseq
from train import resolve_device, metrics_from

ckpt = 'weights/best_mbart_stereotype.pt'
backbone = 'facebook/mbart-large-50'
dev = resolve_device('cuda' if torch.cuda.is_available() else 'cpu')

tok = AutoTokenizer.from_pretrained(backbone)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

net = model_seq2seq.build_model(backbone, pooling='eos', unfreeze_enc=0, unfreeze_dec=2)
net.load_state_dict(torch.load(ckpt, map_location=dev))
net.to(dev).eval()

va = va.reset_index(drop=True)
loader = tseq.make_loader(va, tok, backbone, batch_size=8, context_max_len=256, comment_max_len=128)
out = tseq.collect(net, loader, dev)
m = metrics_from(out['preds'], out['labels'])
print('Overall Val Metrics:', {k: round(v, 4) for k, v in m.items() if k != 'loss'})

# Confusion matrix
cm = confusion_matrix(out['labels'], out['preds'], labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap='Purples')
ax.set_xticks([0, 1]); ax.set_xticklabels(['no', 'yes'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['no', 'yes'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('mBART-50 Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

print('--- Per-Language Validation Results ---')
for lang in sorted(va['lang'].unique()):
    sub = va[va['lang'] == lang].reset_index(drop=True)
    l_loader = tseq.make_loader(sub, tok, backbone, batch_size=8, context_max_len=256, comment_max_len=128)
    l_out = tseq.collect(net, l_loader, dev)
    l_m = metrics_from(l_out['preds'], l_out['labels'])
    print(f"  [{lang}] Accuracy: {l_m['acc']:.4f} | Macro-F1: {l_m['macro_f1']:.4f} | F1-yes: {l_m['f1_yes']:.4f}")


## 6. Alternative Backbone: Meta NLLB-200 (`facebook/nllb-200-distilled-600M`)

NLLB-200 is the state-of-the-art multilingual successor to mBART-50 by Meta (2022). It provides high-capacity multilingual representations and is already used in this repository for translation augmentation.


In [ ]:
!python src/train_seq2seq.py \
    --backbone facebook/nllb-200-distilled-600M \
    --languages EN IT NL \
    --epochs 15 \
    --patience 4 \
    --batch-size 4 \
    --grad-accum 4 \
    --unfreeze-enc 0 \
    --unfreeze-dec 2 \
    --enc-lr 1e-5 \
    --dec-lr 1e-5 \
    --head-lr 3e-4 \
    --pooling eos \
    --out weights/best_nllb_stereotype.pt


## 7. Ablation Quick Screen: EOS vs. Mean Pooling

Test whether EOS token representation or Mean Pooling performs better with the decoder attention stack.


In [ ]:
# Run mean pooling ablation with mBART
!python src/train_seq2seq.py \
    --backbone facebook/mbart-large-50 \
    --languages EN IT NL \
    --epochs 15 \
    --patience 4 \
    --batch-size 4 \
    --grad-accum 4 \
    --unfreeze-enc 0 \
    --unfreeze-dec 2 \
    --pooling mean \
    --out weights/best_mbart_meanpool.pt
